In [20]:
# --- Install dependencies ---
!pip install faiss-cpu "langchain==0.2.11" "langchain-community>=0.2.0" ibm-watsonx-ai transformers sentence-transformers langchain-text-splitters --upgrade

# --- Imports ---
import faiss
import torch
from langchain_text_splitters import CharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain_community.llms import HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, pipeline

# --- Step 1: Load a base model (IBM Granite or fallback HuggingFace model) ---
MODEL_NAME = "ibm-granite/granite-13b-instruct"  # try Granite
FALLBACK_MODEL = "google/flan-t5-base"           # fallback if Granite not accessible

model_loaded_successfully = False
llm_model = None
tokenizer = None
pipeline_task = "text-generation"

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    llm_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="auto")
    model_loaded_successfully = True
except Exception as e:
    print(f"Failed to load {MODEL_NAME}: {e}. Falling back to {FALLBACK_MODEL}")
    try:
        tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL)
        llm_model = AutoModelForSeq2SeqLM.from_pretrained(FALLBACK_MODEL, dtype=torch.float16, device_map="auto")
        pipeline_task = "text-generation" # Changing to text-generation as per latest error message
        model_loaded_successfully = True
    except Exception as inner_e:
        print(f"Failed to load {FALLBACK_MODEL}: {inner_e}. Cannot proceed without a model.")

if not model_loaded_successfully:
    raise SystemExit("Failed to load any language model. Exiting.")

# Determine device to use for pipeline
device = 0 if torch.cuda.is_available() else -1

qa_pipeline = pipeline(pipeline_task, model=llm_model, tokenizer=tokenizer, max_length=512, device=device)
llm = HuggingFacePipeline(pipeline=qa_pipeline)

# --- Step 2: Create sample interview knowledge base ---
documents = [
    "For software engineer interviews, expect data structures, algorithms, system design, and behavioral questions.",
    "HR interviews often focus on teamwork, conflict resolution, and career goals.",
    "For entry-level roles, recruiters look for enthusiasm, learning ability, and basic technical knowledge.",
    "For senior roles, expect leadership, project management, and advanced technical depth.",
    "Behavioral question example: Tell me about a time you resolved a conflict in a team."
]

# --- Step 3: Split and embed documents ---
splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
docs = splitter.create_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)

# --- Step 4: Build Retrieval-Augmented QA system ---
qa = RetrievalQA.from_chain_type(llm=llm, retriever=vectorstore.as_retriever())

# --- Step 5: Example usage ---
def interview_trainer(profile_name, experience_level, job_role, query):
    context = f"Profile: {profile_name}, Experience: {experience_level}, Role: {job_role}. Question: {query}"
    return qa.run(context)

# --- Demo ---
print(interview_trainer("Yamini", "Entry-level", "Software Engineer",
                        "Give me 5 likely interview questions and model answers."))

  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.4-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.3.31-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.3.30-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.3.29-py3-none-any.whl.metadata (2.9 kB)
  Using cached langchain_community-0.3.28-py3-none-any.whl.metadata (2.9 kB)
  Using cached langchain_community-0.3.27-py3-none-any.whl.metadata (2.9 kB)
INFO: pip is still looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"